#### Import Libraries

In [37]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import os

#### Import Data

In [41]:
# Use file path containing extracted CSV files
filepath = r"C:\Users\Hassa\OneDrive - Loughborough University\current\Algorthmic Trading for Beginners\Data Management\datamodules\minute_options_data"          

# List all CSV files that start with 'SPX_minute_options_2023-10-'
csv_files = [f for f in os.listdir(filepath)
            if f.startswith("SPX_minute_options_2023-10-") and f.endswith(".csv")]

print(f"There are {len(csv_files)} csv files in '{filepath}'")

There are 2 csv files in 'C:\Users\Hassa\OneDrive - Loughborough University\current\Algorthmic Trading for Beginners\Data Management\datamodules\minute_options_data'


In [42]:
# Read all CSVs and store in one list
df_list = []
for file in csv_files:
    full_path = os.path.join(filepath, file)
    print(f"Loading {file}")
    df_list.append(pd.read_csv(full_path))

# Combine in one df
options_data_raw  = pd.concat(df_list, ignore_index=True)

print("Done.")
print(options_data_raw.head())

Loading SPX_minute_options_2023-10-02.csv
Loading SPX_minute_options_2023-10-03.csv
Done.
      [QUOTE_DATETIME] [QUOTE_DATE] [QUOTE_TIME] [SYMBOL]  [UNDERLYING_LAST]  \
0  2023-10-02 09:30:00   2023-10-02     09:30:00      SPX            4308.27   
1  2023-10-02 09:30:00   2023-10-02     09:30:00      SPX            4308.27   
2  2023-10-02 09:30:00   2023-10-02     09:30:00      SPX            4308.27   
3  2023-10-02 09:30:00   2023-10-02     09:30:00      SPX            4308.27   
4  2023-10-02 09:30:00   2023-10-02     09:30:00      SPX            4308.27   

  [EXPIRE_DATE] [EXPIRY_TYPE]  [DTE]  [STRIKE]  [STRIKE_DISTANCE]  ...  \
0    2023-10-03         Daily    0.6    4250.0             -58.27  ...   
1    2023-10-03         Daily    0.6    4255.0             -53.27  ...   
2    2023-10-03         Daily    0.6    4260.0             -48.27  ...   
3    2023-10-03         Daily    0.6    4265.0             -43.27  ...   
4    2023-10-03         Daily    0.6    4270.0             

#### Data Cleaning

In [44]:
print(options_data_raw.columns)

Index(['[QUOTE_DATETIME]', '[QUOTE_DATE]', '[QUOTE_TIME]', '[SYMBOL]',
       '[UNDERLYING_LAST]', '[EXPIRE_DATE]', '[EXPIRY_TYPE]', '[DTE]',
       '[STRIKE]', '[STRIKE_DISTANCE]', '[STRIKE_DISTANCE_PCT]', '[C_DELTA]',
       '[C_GAMMA]', '[C_VEGA]', '[C_THETA]', '[C_RHO]', '[C_IV]', '[C_VOLUME]',
       '[C_LAST]', '[C_SIZE]', '[C_BID]', '[C_ASK]', '[P_DELTA]', '[P_GAMMA]',
       '[P_VEGA]', '[P_THETA]', '[P_RHO]', '[P_IV]', '[P_VOLUME]', '[P_LAST]',
       '[P_SIZE]', '[P_BID]', '[P_ASK]'],
      dtype='object')


In [45]:
# Clean column names

options_data_raw.columns = (
                            # This removes spaces from the left
                            options_data_raw.columns.str.lstrip()
                            # This removes spaces from the right
                            .str.rstrip()
                            # This removes the square brackets
                            .str.strip('[]')
                            # This converts all letter to lowercase/
                            .str.lower()
                            )

print(options_data_raw.columns)

Index(['quote_datetime', 'quote_date', 'quote_time', 'symbol',
       'underlying_last', 'expire_date', 'expiry_type', 'dte', 'strike',
       'strike_distance', 'strike_distance_pct', 'c_delta', 'c_gamma',
       'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last', 'c_size',
       'c_bid', 'c_ask', 'p_delta', 'p_gamma', 'p_vega', 'p_theta', 'p_rho',
       'p_iv', 'p_volume', 'p_last', 'p_size', 'p_bid', 'p_ask'],
      dtype='object')


In [47]:
# Convert 'quote_time' to a datetime dtype and set it as df index so the df is time-indexed
options_data_raw.index = pd.to_datetime(options_data_raw.quote_date)

# Remove index name
options_data_raw.index.name = ''

options_data_raw.tail(4)

,quote_datetime,quote_date,quote_time,symbol,underlying_last,expire_date,expiry_type,dte,strike,strike_distance,...,p_gamma,p_vega,p_theta,p_rho,p_iv,p_volume,p_last,p_size,p_bid,p_ask
,,,,,,,,,,,,,,,,,,,,,
2023-10-03,2023-10-03 16:00:00,2023-10-03,16:00:00,SPX,4307.86,2023-10-21,Monthly,17.33,4425.0,117.14,...,0.00003,179.40204,-2.38374,-21.02649,1.74753,29,1547.73,2 x 18,1406.22,1703.47
2023-10-03,2023-10-03 16:00:00,2023-10-03,16:00:00,SPX,4307.86,2023-10-21,Monthly,17.33,4450.0,142.14,...,0.00002,179.40204,-3.30286,-25.64075,2.46057,29,1565.51,9 x 15,1423.27,1721.96
2023-10-03,2023-10-03 16:00:00,2023-10-03,16:00:00,SPX,4307.86,2023-10-21,Monthly,17.33,4475.0,167.14,...,0.00001,179.40204,-4.39910,-30.29556,3.31112,26,1583.59,1 x 16,1440.62,1740.75
2023-10-03,2023-10-03 16:00:00,2023-10-03,16:00:00,SPX,4307.86,2023-10-21,Monthly,17.33,4500.0,192.14,...,0.00001,179.40204,-5.67246,-34.98905,4.29917,21,1601.98,9 x 4,1458.26,1759.86


In [48]:
# List of useless columns to drop
cols_to_drop = [
                'quote_date',
                'quote_datetime',
                'quote_time'             
]

# Use .drop() to remove cols
options_data_raw.drop(columns=cols_to_drop, inplace=True)
options_data_raw.columns

Index(['symbol', 'underlying_last', 'expire_date', 'expiry_type', 'dte',
       'strike', 'strike_distance', 'strike_distance_pct', 'c_delta',
       'c_gamma', 'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last',
       'c_size', 'c_bid', 'c_ask', 'p_delta', 'p_gamma', 'p_vega', 'p_theta',
       'p_rho', 'p_iv', 'p_volume', 'p_last', 'p_size', 'p_bid', 'p_ask'],
      dtype='object')

In [49]:
# Check if data values have consistency issues
print(options_data_raw['c_iv'][-1])
print(type(options_data_raw['c_iv'][-1]))
print(options_data_raw['strike_distance'][-1])
print(type(options_data_raw['strike_distance'][-1]))
print(options_data_raw['c_delta'][-1])
print(type(options_data_raw['c_delta'][-1]))

3.92997
<class 'numpy.float64'>
192.14
<class 'numpy.float64'>
0.44665
<class 'numpy.float64'>


In [50]:
# Everything seems in order but lets apply a clean up just in case for l and r spaces for all str cols
options_data_raw = options_data_raw.applymap(lambda x: x.strip()
                                             if isinstance(x, str) 
                                             else x)

# Check
print(options_data_raw['dte'][-1])
print(type(options_data_raw['dte'][-1]))
print(options_data_raw['underlying_last'][-1])
print(type(options_data_raw['underlying_last'][-1]))
print(options_data_raw['expiry_type'][-1])
print(type(options_data_raw['expiry_type'][-1]))

17.33
<class 'numpy.float64'>
4307.86
<class 'numpy.float64'>
Monthly
<class 'str'>


In [53]:
# Check dtypes of all cols
options_data_raw.dtypes.value_counts()

float64    23
object      5
int64       2
Name: count, dtype: int64

In [54]:
options_data_raw.dtypes

symbol                  object
underlying_last        float64
expire_date             object
expiry_type             object
dte                    float64
strike                 float64
strike_distance        float64
strike_distance_pct    float64
c_delta                float64
c_gamma                float64
c_vega                 float64
c_theta                float64
c_rho                  float64
c_iv                   float64
c_volume                 int64
c_last                 float64
c_size                  object
c_bid                  float64
c_ask                  float64
p_delta                float64
p_gamma                float64
p_vega                 float64
p_theta                float64
p_rho                  float64
p_iv                   float64
p_volume                 int64
p_last                 float64
p_size                  object
p_bid                  float64
p_ask                  float64
dtype: object

In [64]:
# Identify cols based on their naming patterns
float_cols = [
              col for col in options_data_raw.columns
              if col.startswith(("c_", "p_")) and col not in ("c_size", "p_size")
]

date_cols = [col for col in options_data_raw.columns
             if "date" in col.lower()
]

# Convert numeric cols
for col in float_cols:
    options_data_raw[col] = pd.to_numeric(options_data_raw[col], errors='coerce').astype("float64")

# Convert date cols
for col in date_cols:
    options_data_raw[col] = pd.to_datetime(options_data_raw[col], errors='coerce')
    
    

In [65]:
print("Converted numeric columns:", float_cols)
print("Converted date columns:", date_cols)

Converted numeric columns: ['c_delta', 'c_gamma', 'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last', 'c_bid', 'c_ask', 'p_delta', 'p_gamma', 'p_vega', 'p_theta', 'p_rho', 'p_iv', 'p_volume', 'p_last', 'p_bid', 'p_ask']
Converted date columns: ['expire_date']


In [66]:
options_data_raw.dtypes.value_counts()

float64           25
object             4
datetime64[ns]     1
Name: count, dtype: int64

In [67]:
options_data_raw.dtypes

symbol                         object
underlying_last               float64
expire_date            datetime64[ns]
expiry_type                    object
dte                           float64
strike                        float64
strike_distance               float64
strike_distance_pct           float64
c_delta                       float64
c_gamma                       float64
c_vega                        float64
c_theta                       float64
c_rho                         float64
c_iv                          float64
c_volume                      float64
c_last                        float64
c_size                         object
c_bid                         float64
c_ask                         float64
p_delta                       float64
p_gamma                       float64
p_vega                        float64
p_theta                       float64
p_rho                         float64
p_iv                          float64
p_volume                      float64
p_last      

#### Export Cleaned Data


In [69]:
options_data_raw.to_csv(r"C:\Users\Hassa\OneDrive - Loughborough University\current\Algorthmic Trading for Beginners\Data Management\datamodules\minute_options_data\options_data_raw_cleaned.csv")